# Pearson / Spearman / MI：関係形状ごとの比較

同じ目的変数 `Y` に対して、4種類のセンサ `X` を人工的に作り、
**Pearson相関 / Spearman相関 / Mutual Information (MI)** がどう反応するか比較します。

対象:
- `X_linear`: 線形・単調増加
- `X_monotonic_nonlinear`: 非線形・単調増加
- `X_u_shape`: U字型
- `X_cliff`: ガケ型（閾値で急変）

> これは因果関係の教材ではなく、**X-Yの関係形状に対して各指標がどう反応するか**を見る教材です。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

try:
    from sklearn.feature_selection import mutual_info_regression
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("scikit-learn が無いため MI_sklearn は NaN になります。")


## 1. 同じYに対して4種類のXを作る

まず `X_cliff` から「ガケ」を持つ `Y` を作り、そのYから別のXを構成します。
これにより、**同じYに対して関係形状だけが異なるセンサX** を比較できます。

- 線形: `X ≈ Y`
- 非線形単調: `X = scaled(Y)^12`（順位は完全に保つが直線ではない）
- U字: `Y ≈ X^2` になるよう正負ランダムなXを作る
- ガケ: Xが閾値を超えるとYが急にジャンプ


In [ ]:
rng = np.random.default_rng(42)
n = 5000

# ガケ型の基準センサ
X_cliff = rng.uniform(-1, 1, n)

# 閾値0.15を超えるとYが急に+2.5ジャンプ
Y = X_cliff + 2.5 * (X_cliff > 0.15).astype(float) + rng.normal(0, 0.08, n)

# 0〜1へスケーリング（他の関係形状を作りやすくするため）
Y_scaled = (Y - Y.min()) / (Y.max() - Y.min())

# 線形・単調増加
X_linear = Y + rng.normal(0, 0.15, n)

# 非線形・単調増加
# 単調変換なので順位は完全に保存されるが、一次直線関係からは外れる
X_monotonic_nonlinear = Y_scaled ** 12

# U字型: Y ≈ X^2 の形を作る
sign = rng.choice([-1, 1], size=n)
X_u_shape = sign * np.sqrt(Y_scaled) + rng.normal(0, 0.02, n)

df = pd.DataFrame({
    "Y": Y,
    "X_linear": X_linear,
    "X_monotonic_nonlinear": X_monotonic_nonlinear,
    "X_u_shape": X_u_shape,
    "X_cliff": X_cliff,
})

df.head()


## 2. 散布図で形を確認

EDAでは、相関係数やMIだけで判断せず、まず散布図で「形」を確認します。


In [ ]:
features = [
    "X_linear",
    "X_monotonic_nonlinear",
    "X_u_shape",
    "X_cliff",
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, feature in zip(axes.ravel(), features):
    ax.scatter(df[feature], df["Y"], s=8, alpha=0.25)
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel("Y")

plt.tight_layout()
plt.show()


## 3. 3指標を計算

### Pearson相関
\[
r = \frac{\mathrm{Cov}(X,Y)}{\sigma_X\sigma_Y}
\]
**一次直線的な共変動**を見る。

### Spearman相関
\[
\rho_s = \mathrm{Corr}(\mathrm{rank}(X),\mathrm{rank}(Y))
\]
順位に変えてからPearson相関を取るため、**単調増加・単調減少**に強い。

### Mutual Information
離散化した場合の式は
\[
I(X;Y)=\sum_{x,y}p(x,y)\log\frac{p(x,y)}{p(x)p(y)}
\]
XとYが独立なら `p(x,y)=p(x)p(y)` なのでMIは0。
U字やガケのように相関だけでは表しにくい依存も検出できる。


In [ ]:
def histogram_mi(x, y, bins=20):
    """学習用: 2次元ヒストグラムからMIを直接計算する簡易推定。"""
    hist_xy, _, _ = np.histogram2d(x, y, bins=bins)
    pxy = hist_xy / hist_xy.sum()
    px = pxy.sum(axis=1, keepdims=True)
    py = pxy.sum(axis=0, keepdims=True)
    px_py = px @ py

    mask = pxy > 0
    return np.sum(pxy[mask] * np.log(pxy[mask] / px_py[mask]))


def calculate_metrics(df, y_col, features):
    rows = []

    for feature in features:
        x = df[feature].to_numpy()
        y = df[y_col].to_numpy()

        pearson_r, pearson_p = pearsonr(x, y)
        spearman_rho, spearman_p = spearmanr(x, y)
        mi_hist = histogram_mi(x, y, bins=20)

        if SKLEARN_AVAILABLE:
            mi_knn = mutual_info_regression(
                x.reshape(-1, 1),
                y,
                random_state=42
            )[0]
        else:
            mi_knn = np.nan

        rows.append({
            "feature": feature,
            "Pearson_r": pearson_r,
            "Spearman_rho": spearman_rho,
            "MI_histogram": mi_hist,
            "MI_sklearn": mi_knn,
            "Pearson_p": pearson_p,
            "Spearman_p": spearman_p,
        })

    return pd.DataFrame(rows)


result = calculate_metrics(df, "Y", features)
result


## 4. 比較しやすい表にする

- Pearson / Spearman は `-1〜1` で方向を持つ
- MI は `0以上` で方向を持たない
- MIの絶対値には「0.7なら強い」のような共通基準はないため、同じデータ内のランキングとして使うのが実務的


In [ ]:
comparison = result.copy()
comparison["abs_Pearson"] = comparison["Pearson_r"].abs()
comparison["abs_Spearman"] = comparison["Spearman_rho"].abs()

comparison[
    [
        "feature",
        "Pearson_r",
        "Spearman_rho",
        "abs_Pearson",
        "abs_Spearman",
        "MI_histogram",
        "MI_sklearn",
    ]
].sort_values("MI_sklearn", ascending=False)


## 5. 指標ごとの強さを可視化


In [ ]:
plot_df = comparison.set_index("feature")[[
    "abs_Pearson",
    "abs_Spearman",
    "MI_sklearn"
]]

axes = plot_df.plot(
    kind="bar",
    subplots=True,
    figsize=(11, 10),
    legend=False,
    title=["|Pearson|", "|Spearman|", "Mutual Information"],
)

for ax in axes:
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


## 6. 読み方

| パターン | Pearson | Spearman | MI | なぜ？ |
|---|---|---|---|---|
| 線形・単調 | 高い | 高い | 高い | 直線・順位・依存のすべてがある |
| 非線形・単調 | Pearsonは低下しうる | 高い | 高い | 順位は保つが直線ではない |
| U字 | 0付近になりやすい | 0付近になりやすい | 高い | 左右で相関が相殺されてもXを知ればYが分かる |
| ガケ | 中〜高 | 高くなりうる | 高い | 閾値でYの分布が大きく変わる |

### 実務でのポイント

1. **Pearson**: 「一次直線として効いているか？」
2. **Spearman**: 「一貫して増える / 減るか？」
3. **MI**: 「形は問わず、Xを知ることでYについて情報が増えるか？」

したがって大量センサの単変量EDAでは、3つを並べると関係の種類を推測しやすくなります。

特に、

- `Pearson ≈ Spearman ≫ 0` → 線形・単調候補
- `Spearman ≫ Pearson` → 単調非線形候補
- `Pearson ≈ 0, Spearman ≈ 0, MIが高い` → U字など非単調候補
- `MIが高い + 散布図に急変` → ガケ・閾値候補

という見方ができます。

> 注意: これらはスクリーニングです。実データではTool、Product、Recipe、時間などの交絡を確認し、LightGBM/SHAP、GAM/ALE、階層モデルなどで再検証します。


## 7. 実データへ置き換える

```python
df = pd.read_csv("your_sensor_data.csv")

y_col = "yield"
features = [
    c for c in df.select_dtypes(include="number").columns
    if c != y_col
]

result = calculate_metrics(df.dropna(), y_col, features)
```

MIはサンプル数・ノイズ・推定法・bin数で値が変わるため、絶対値だけでなく散布図や他指標と合わせて解釈してください。
